# NumPy Optimization: Vectorization and the C Backend

This notebook is a **from-scratch to intermediate** guide to **optimizing NumPy code** by
exploiting **vectorization** and how NumPy uses its **C backend** for array operations.

We cover:

1. **Why vectorization wins**: Python loops vs array ops (work runs in C)
2. **Broadcasting**: shape rules so you write vectorized code without explicit loops
3. **Memory layout**: contiguous arrays, strides, views vs copies
4. **Dtypes**: choosing the right type for speed and memory
5. **Reducing temporaries**: in-place ops and expression fusion
6. **Using the right primitives**: dot, matmul, einsum, and BLAS-backed operations
7. **A peek at the C backend**: how NumPy arrays map to C memory; when copies happen
8. **Best practices and common pitfalls**

The goal is to write **fewer Python loops** and **more array-level operations** so that
the heavy work runs in NumPy’s compiled C (and optionally BLAS/LAPACK) code.

### Contents

- [1. Why Vectorization Wins: Loops vs Array Ops](#1-why-vectorization-wins-loops-vs-array-ops)
- [2. Broadcasting: Shape Rules for Vectorized Code](#2-broadcasting-shape-rules-for-vectorized-code)
- [3. Memory Layout: Contiguous Arrays, Strides, and Views](#3-memory-layout-contiguous-arrays-strides-and-views)
- [4. Dtypes: Right Type for Speed and Memory](#4-dtypes-right-type-for-speed-and-memory)
- [5. Reducing Temporaries: In-Place and Expression Shape](#5-reducing-temporaries-in-place-and-expression-shape)
- [6. Using the Right Primitives: dot, matmul, einsum](#6-using-the-right-primitives-dot-matmul-einsum)
- [7. Peek at the C Backend: How NumPy Arrays Map to C](#7-peek-at-the-c-backend-how-numpy-arrays-map-to-c)
- [8. Best Practices and Common Pitfalls](#8-best-practices-and-common-pitfalls)
- [9. Putting It Together: Rolling Mean (Loop vs Vectorized)](#9-putting-it-together-rolling-mean-loop-vs-vectorized)

## 1. Why Vectorization Wins: Loops vs Array Ops

NumPy’s core is implemented in **C**. When you write **array-level operations** (e.g. `a + b`,
`np.sum(x)`), Python calls into that C code once; the loop over elements runs entirely
in C. When you write a **Python loop** over elements, each iteration does Python bytecode,
type checks, and function calls — so it’s much slower.

**Vectorization** means expressing your algorithm as operations on whole arrays (or slices)
instead of element-by-element in Python. The “backend” that runs is NumPy’s C implementation
(and often optimized BLAS/LAPACK for linear algebra).

In [ ]:
# 1.1 Python loop vs vectorized: same math, different backend

import numpy as np
import time

n = 1_000_000
a = np.random.randn(n).astype(np.float64)
b = np.random.randn(n).astype(np.float64)

# Python loop: each add runs in Python interpreter (slow)
def loop_add(x, y):
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i] + y[i]
    return out

# Vectorized: one call into NumPy's C backend; loop over elements runs in C (fast)
def vectorized_add(x, y):
    return x + y

# Timing (vectorized is orders of magnitude faster)
t0 = time.perf_counter()
c_loop = loop_add(a, b)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
c_vec = vectorized_add(a, b)
t_vec = time.perf_counter() - t0

print(f"Loop:      {t_loop*1000:.2f} ms")
print(f"Vectorized: {t_vec*1000:.2f} ms")
print(f"Speedup:   {t_loop/t_vec:.0f}x")
print("Results match:", np.allclose(c_loop, c_vec))

In [ ]:
# 1.2 Sum and element-wise math: keep the loop in C

# Sum
t0 = time.perf_counter()
s_loop = sum(a[i] for i in range(len(a)))  # pure Python
t_loop = time.perf_counter() - t0
t0 = time.perf_counter()
s_vec = np.sum(a)  # single C-level reduction
t_vec = time.perf_counter() - t0
print(f"Sum: loop {t_loop*1000:.0f} ms  vs  vectorized {t_vec*1000:.2f} ms")

# Element-wise function (e.g. "sigmoid" style: 1/(1+exp(-x)) approximated for demo)
def loop_sigmoid(x):
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = 1.0 / (1.0 + np.exp(-x[i]))  # still one Python call per element!
    return out

def vec_sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))  # entire array in one C-friendly path

t0 = time.perf_counter()
loop_sigmoid(a)
t_loop = time.perf_counter() - t0
t0 = time.perf_counter()
vec_sigmoid(a)
t_vec = time.perf_counter() - t0
print(f"Sigmoid: loop {t_loop*1000:.0f} ms  vs  vectorized {t_vec*1000:.2f} ms")

## 2. Broadcasting: Shape Rules for Vectorized Code

**Broadcasting** lets NumPy combine arrays of different shapes without writing loops:
dimensions are compared from the trailing end; they must match or one must be 1. NumPy
“extends” the smaller array along that dimension (conceptually; often no copy is made).
This is how you write compact, fast code like `prices - moving_avg` where `moving_avg`
is 1D and `prices` is 1D with the same length — or a scalar with a whole array.

In [ ]:
# 2.1 Broadcasting examples: no explicit loop

# Scalar with array
prices = np.array([100.0, 101.0, 99.0])
print("prices - 100:", prices - 100)  # 100 broadcast to shape (3,)

# (3,) with (3,) — element-wise
ma = np.array([100.5, 100.2, 99.8])
print("prices - ma:", prices - ma)

# (4, 3) with (3,): each row gets the same (3,) subtracted
X = np.random.randn(4, 3)
row_means = X.mean(axis=1, keepdims=True)  # shape (4, 1)
print("X - row_means shape:", (X - row_means).shape)  # (4, 3); (4,1) broadcasts to (4,3)

# Typical: center each column (subtract column mean)
col_means = X.mean(axis=0)  # (3,)
X_centered = X - col_means  # (4,3) - (3,) -> broadcast
print("Column means of X_centered:", X_centered.mean(axis=0))

## 3. Memory Layout: Contiguous Arrays, Strides, and Views

NumPy arrays in C are a block of memory plus **strides**: how many bytes to step in each
dimension. **C-contiguous** (row-major) means the last dimension is contiguous in memory;
**F-contiguous** (column-major) means the first dimension is. Many NumPy operations
expect C-contiguous input and may **copy** if given a non-contiguous or transposed view.
Understanding this helps you avoid hidden copies and choose the right layout for
external C/Fortran code.

In [ ]:
# 3.1 Inspecting layout: flags and strides

A = np.arange(12).reshape(3, 4)
print("Shape:", A.shape, "  Strides (bytes):", A.strides)
print("C-contiguous:", A.flags["C_CONTIGUOUS"], "  F-contiguous:", A.flags["F_CONTIGUOUS"])

# Transpose is a view (different strides, same data)
AT = A.T
print("A.T strides:", AT.strides, "  C-contiguous:", AT.flags["C_CONTIGUOUS"])

# Slicing: often a view (no copy)
B = A[1:, :2]
print("A[1:, :2] is view:", B.base is A or B.base is A.base)

# Forcing C-contiguous when needed (copies if not already)
Ac = np.ascontiguousarray(AT)
print("ascontiguousarray(A.T) is copy:", Ac.base is not None and Ac.base is not AT)

## 4. Dtypes: Right Type for Speed and Memory

Using a smaller or more hardware-friendly **dtype** can reduce memory bandwidth and
improve cache use. `float32` is half the size of `float64`; integer types (`int32`,
`int64`) avoid float math when you only need integers. NumPy’s C backend is optimized
for these native types; avoid `object` or arbitrary Python types in hot paths.

In [ ]:
# 4.1 float32 vs float64: memory and speed

n = 5_000_000
a64 = np.random.randn(n).astype(np.float64)
a32 = np.random.randn(n).astype(np.float32)

print("float64 size (MB):", a64.nbytes / 2**20)
print("float32 size (MB):", a32.nbytes / 2**20)

# Same vectorized op; float32 can be faster due to memory bandwidth / SIMD
t0 = time.perf_counter()
_ = a64 + a64 * 2.0
t64 = time.perf_counter() - t0
t0 = time.perf_counter()
_ = a32 + a32 * 2.0
t32 = time.perf_counter() - t0
print(f"Add+scale: float64 {t64*1000:.2f} ms  float32 {t32*1000:.2f} ms")

## 5. Reducing Temporaries: In-Place and Expression Shape

Expressions like `a = a + b` create a **new** array and then rebind `a`. Using
`a += b` (in-place) or `np.add(a, b, out=a)` reuses the buffer of `a`, reducing
allocations. For long chains, temporary arrays can dominate cost; combining
operations (e.g. one loop that does several steps) or using `out=` where available
keeps the C backend busy without extra memory traffic.

In [ ]:
# 5.1 In-place and out= to avoid extra allocations

x = np.ones(1_000_000, dtype=np.float64)

# Creates a new array, then x is rebound
# x = x + 1.0

# Modifies x in place (no new array for result)
x += 1.0
np.multiply(x, 2.0, out=x)  # x *= 2.0 same idea

# Many ufuncs accept out=
y = np.empty_like(x)
np.add(x, np.sin(x), out=y)  # write result into preallocated y
print("Sample y:", y[:3])

## 6. Using the Right Primitives: dot, matmul, einsum

NumPy delegates **matrix multiplication** and many linear-algebra ops to **BLAS/LAPACK**
(libraries optimized in C/Fortran). So `np.dot(a, b)`, `np.matmul`, and `@` for
matrices use this backend. Prefer these over manual Python loops. **einsum** is
another way to express sums over indices and can be very efficient for certain
patterns (e.g. batched matmuls, traces). Choosing the right op keeps the hot path
in compiled, often SIMD/BLAS-optimized code.

In [ ]:
# 6.1 Matrix multiply: loop vs BLAS-backed @

M, K, N = 200, 200, 200
A = np.random.randn(M, K).astype(np.float64)
B = np.random.randn(K, N).astype(np.float64)

def loop_matmul(A, B):
    out = np.zeros((A.shape[0], B.shape[1]))
    for i in range(A.shape[0]):
        for j in range(B.shape[1]):
            out[i, j] = np.dot(A[i, :], B[:, j])
    return out

t0 = time.perf_counter()
loop_matmul(A, B)
t_loop = time.perf_counter() - t0
t0 = time.perf_counter()
C = A @ B  # BLAS-backed
t_blas = time.perf_counter() - t0
print(f"Matmul {M}x{K} @ {K}x{N}: loop {t_loop*1000:.0f} ms  @ (BLAS) {t_blas*1000:.2f} ms")

# einsum: compact and often optimized (e.g. "ik,kj->ij" for matmul)
t0 = time.perf_counter()
C2 = np.einsum("ik,kj->ij", A, B)
print(f"einsum same: {(time.perf_counter()-t0)*1000:.2f} ms  same result: {np.allclose(C, C2)}")

## 7. Peek at the C Backend: How NumPy Arrays Map to C

NumPy arrays are **wrapper objects** around a C-style block of memory (plus shape, strides,
dtype). The **NumPy C API** and protocols like **`__array_interface__`** expose this so
that C/Cython/Fortran code can read and write the same memory without copying. In Python
you rarely need to touch this directly, but it explains why:

- **Vectorized ops** are fast: they iterate over this single block in C.
- **Views** (slices, transpose) just change shape/strides; they don’t copy the data.
- **Copying** happens when you need a new buffer (e.g. `arr.copy()`, or a result that
  can’t be laid out in the same strides).

Knowing when NumPy creates a copy helps you avoid unnecessary allocations in hot paths.

In [ ]:
# 7.1 Array interface and when copies happen

z = np.arange(6, dtype=np.float64).reshape(2, 3)
# __array_interface__ exposes shape, strides, typestr, data (pointer) for C interop
print("Shape:", z.__array_interface__["shape"])
print("Strides:", z.__array_interface__["strides"])
print("Data pointer (id):", z.__array_interface__["data"][0])

# Views share the same base buffer
z_view = z[1, :]
print("z_view.base is z:", z_view.base is z)

# These create copies: different buffer
z_copy = z.copy()
z_from_list = np.array(z.tolist())
print("z.copy() shares base:", z_copy.base is z)
print("np.array(z.tolist()) shares base:", z_from_list.base is z)

## 8. Best Practices and Common Pitfalls

**Do**

- Prefer **array-level operations** and **broadcasting** over Python loops.
- Use a **suitable dtype** (e.g. `float32` if precision allows) to save memory and bandwidth.
- Use **in-place** (`+=`, `*=` or `out=`) when you don’t need to keep the old array.
- Use **`@`**, **`np.dot`**, **`np.einsum`** for linear algebra so BLAS is used.
- Prefer **views** (slicing, reshape with same size) over copies when possible.

**Avoid**

- **Python loops** over array elements for numeric work.
- **Mixing dtypes** or using `object` arrays in hot paths.
- **Unnecessary copies**: e.g. `np.array(arr)` when `arr` is already an array; check if you need `arr.copy()`.
- **Repeated small allocations** in a loop; preallocate and reuse buffers.

**Enhancing vectorization**

- If an operation isn’t in NumPy, consider **Numba** or **Cython** for the hot loop while
  keeping the rest in NumPy (see the Numba and Cython notebooks).
- **NumPy’s C backend** is already optimized; your leverage is to **stay in that backend**
  by writing vectorized code and avoiding Python-level loops and copies.

## 9. Putting It Together: Rolling Mean (Loop vs Vectorized)

A **rolling mean** over a 1D array can be done with a Python loop (slow) or with
**cumulative sum** and array slicing (fast): `(cumsum[i] - cumsum[i-w]) / w`. This
keeps everything in the C backend and avoids per-element Python.

In [ ]:
# 9.1 Rolling mean: loop vs cumsum vectorization

def rolling_mean_loop(x, w):
    n = len(x)
    out = np.full(n, np.nan)
    for i in range(w - 1, n):
        out[i] = np.mean(x[i - w + 1 : i + 1])
    return out


def rolling_mean_cumsum(x, w):
    cs = np.cumsum(np.concatenate([[0], x.astype(np.float64)]))
    out = np.full(len(x), np.nan)
    out[w - 1 :] = (cs[w:] - cs[:-w]) / w
    return out


x = 1.2345 + np.random.randn(50_000) * 0.0005
w = 20
t0 = time.perf_counter()
rolling_mean_loop(x, w)
t_loop = time.perf_counter() - t0
t0 = time.perf_counter()
rolling_mean_cumsum(x, w)
t_vec = time.perf_counter() - t0
print(f"Rolling mean (w={w}): loop {t_loop*1000:.0f} ms  cumsum {t_vec*1000:.2f} ms  speedup {t_loop/t_vec:.0f}x")
print("Match:", np.allclose(rolling_mean_loop(x, w), rolling_mean_cumsum(x, w), equal_nan=True))